# Examination Timetable Scheduling Using Particle Swarm Optimization

**Nature-Inspired Computing · Experiment 9 Mini Project**  
**Team:** Hardik Hinduja (20), Avinash Gehi (11), Sahil Deshmukh (9), Siddharth Dey (10)

This self-contained notebook creates a reproducible synthetic examination instance, builds a weighted student-conflict graph, runs a discrete Particle Swarm Optimization (PSO) search with a timetable repair decoder, and reports detailed plain tables. No real student records are used. Run all cells from top to bottom in Google Colab.

**Chosen topic:** Project 2, Examination Timetable Scheduling Using PSO. It has an explicit measurable objective: minimize simultaneous exams taken by the same student, then use fewer periods with balanced loads.

## Guardrails
- Synthetic registrations only; do not paste student names, IDs, grades, or private registration data into this demonstration.
- Every exam is assigned exactly one period within the declared timetable horizon.
- A post-run audit checks student conflicts, exam count, period bounds, and schedule completeness.
- A zero-conflict result is reported only when the independent audit verifies it.
- The fixed seed makes the generated instance and search reproducible; it is not evidence about a real institution's timetable.

## The prompt used to scope the implementation
"Build a complete, reproducible Nature-Inspired Computing mini project for examination timetable scheduling with discrete PSO. Generate a clearly synthetic student-course registration instance; construct a weighted conflict graph; implement standard personal/global-best particle updates plus a deterministic repair decoder; compare PSO with random assignment and a degree-greedy baseline; audit hard constraints and student clashes; display clear monochrome tables for parameters, benchmark metrics, final schedule, per-exam registration totals, sample registrations, and conflict weights; export CSV outputs; explain limitations and privacy guardrails; and provide a minimal black-on-white interactive website with a working 3D schedule visualization. Do not present synthetic results as real institutional data."


## 1. Imports and reproducibility
We use the Python standard library for generation and optimization, plus pandas and matplotlib for readable result tables and a monochrome conflict graph.

In [ ]:
import math, random, time
from datetime import date, timedelta
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
pd.set_option('display.max_rows', 50)
pd.set_option('display.max_columns', 20)
pd.set_option('display.width', 140)


## 2. Complete scheduling implementation
Encoding: one real-valued period preference per exam. The decoder orders highly constrained exams first, assigns the period with the lowest incremental conflict penalty, then uses load and particle preference as tie-breakers. Fitness is `clashes × 10,000 + occupied periods × 100 + load spread`.

In [ ]:
"""Reproducible discrete PSO exam scheduler used in the Colab report."""
from __future__ import annotations

import math
import random
import time
from typing import Any


def generate_instance(seed: int = 42, students: int = 120, exams: int = 12):
    """Create a clearly synthetic student-to-exam registration sample."""
    rng = random.Random(seed)
    records: list[list[int]] = []
    for student in range(students):
        # Three synthetic cohorts each take courses from their own four-exam
        # block, keeping the conflict graph sparse and the demo timetable feasible.
        cohort_count = 3
        cohort_size = math.ceil(students / cohort_count)
        cohort = min(student // cohort_size, cohort_count - 1)
        block_start = cohort * (exams // cohort_count)
        courses = list(range(block_start, min(block_start + exams // cohort_count, exams)))
        rng.shuffle(courses)
        load = min(len(courses), 3 + (student % 4 == 0))
        records.append(sorted(courses[:load]))
    sizes = [sum(exam in row for row in records) for exam in range(exams)]
    weights = [[0] * exams for _ in range(exams)]
    for row in records:
        for i, left in enumerate(row):
            for right in row[i + 1:]:
                weights[left][right] += 1
                weights[right][left] += 1
    return records, sizes, weights


def conflict_count(plan: list[int], weights: list[list[int]]) -> int:
    return sum(weights[a][b] for a in range(len(plan)) for b in range(a + 1, len(plan))
               if plan[a] == plan[b])


def evaluate(plan: list[int], weights: list[list[int]], periods: int) -> dict[str, Any]:
    clashes = conflict_count(plan, weights)
    loads = [plan.count(period) for period in range(periods)]
    occupied = [load for load in loads if load]
    spread = math.sqrt(sum((load - sum(occupied) / len(occupied)) ** 2 for load in occupied)
                       / len(occupied)) if occupied else 0.0
    return {"plan": plan[:], "clashes": clashes, "periods_used": len(occupied),
            "load_spread": spread, "loads": loads,
            "fitness": clashes * 10_000 + len(occupied) * 100 + spread}


def decode_repair(position: list[float], weights: list[list[int]], periods: int) -> list[int]:
    """Decode a particle, placing high-conflict exams first and repairing clashes."""
    exams = len(position)
    degree = [sum(row) for row in weights]
    order = sorted(range(exams), key=lambda e: (-degree[e], position[e], e))
    plan = [-1] * exams
    loads = [0] * periods
    for exam in order:
        choices = []
        for period in range(periods):
            clash_cost = sum(weights[exam][other] for other in range(exams)
                             if plan[other] == period)
            score = (clash_cost * 10_000 + (20 if not loads[period] else 0)
                     + loads[period] * 3 + abs(period - round(position[exam])) * 0.02)
            choices.append((score, period))
        chosen = min(choices)[1]
        plan[exam] = chosen
        loads[chosen] += 1
    return plan


def optimize_pso(weights: list[list[int]], periods: int = 8, seed: int = 141,
                 particles: int = 32, iterations: int = 100):
    """PSO over real-valued period preferences with a discrete repair decoder."""
    rng = random.Random(seed)
    dimensions = len(weights)
    swarm = []
    global_best = None
    for _ in range(particles):
        position = [rng.random() * (periods - 1) for _ in range(dimensions)]
        velocity = [0.0] * dimensions
        plan = decode_repair(position, weights, periods)
        fit = evaluate(plan, weights, periods)
        particle = {"position": position, "velocity": velocity,
                    "best_position": position[:], "best_fitness": fit["fitness"]}
        swarm.append(particle)
        if global_best is None or fit["fitness"] < global_best["fitness"]:
            global_best = {**fit, "position": position[:]}

    for _iteration in range(iterations):
        for particle in swarm:
            for d in range(dimensions):
                r1, r2 = rng.random(), rng.random()
                particle["velocity"][d] = (
                    0.68 * particle["velocity"][d]
                    + 1.42 * r1 * (particle["best_position"][d] - particle["position"][d])
                    + 1.42 * r2 * (global_best["position"][d] - particle["position"][d])
                )
                particle["position"][d] = max(0.0, min(periods - 1,
                    particle["position"][d] + particle["velocity"][d]))
            plan = decode_repair(particle["position"], weights, periods)
            fit = evaluate(plan, weights, periods)
            if fit["fitness"] < particle["best_fitness"]:
                particle["best_fitness"] = fit["fitness"]
                particle["best_position"] = particle["position"][:]
            if fit["fitness"] < global_best["fitness"]:
                global_best = {**fit, "position": particle["position"][:]}
    return {**global_best, "particles": particles, "iterations": iterations}


def random_baseline(weights: list[list[int]], periods: int = 8, seed: int = 141):
    rng = random.Random(seed)
    plan = [rng.randrange(periods) for _ in weights]
    return evaluate(plan, weights, periods)


def greedy_baseline(weights: list[list[int]], periods: int = 8):
    dimension = len(weights)
    degree = [sum(row) for row in weights]
    order = sorted(range(dimension), key=lambda e: -degree[e])
    plan = [-1] * dimension
    loads = [0] * periods
    for exam in order:
        scores = []
        for period in range(periods):
            clash_cost = sum(weights[exam][other] for other in range(dimension)
                             if plan[other] == period)
            scores.append((clash_cost * 10_000 + (20 if not loads[period] else 0)
                           + loads[period] * 3, period))
        chosen = min(scores)[1]
        plan[exam] = chosen
        loads[chosen] += 1
    return evaluate(plan, weights, periods)


def run_experiment(seed: int = 42, students: int = 120, exams: int = 12,
                   periods: int = 8, particles: int = 32, iterations: int = 100):
    records, sizes, weights = generate_instance(seed, students, exams)
    results = {"Random assignment": random_baseline(weights, periods),
               "Degree-greedy repair": greedy_baseline(weights, periods),
               "Discrete PSO": optimize_pso(weights, periods, seed + 99, particles, iterations)}
    conflicts = sum(weights[a][b] for a in range(exams) for b in range(a + 1, exams))
    return {"records": records, "sizes": sizes, "weights": weights, "results": results,
            "parameters": {"seed": seed, "students": students, "exams": exams,
                           "periods": periods, "particles": particles, "iterations": iterations,
                           "total_conflict_weight": conflicts}}


def build_schedule_table(result: dict[str, Any], periods: int):
    from datetime import date, timedelta
    start = date(2026, 11, 9)
    plan = result["plan"]
    rows = []
    for exam, period in enumerate(plan):
        day = period // 2
        slot = "09:00-12:00" if period % 2 == 0 else "13:00-16:00"
        rows.append({"Exam": f"EX-{exam + 1:02d}", "Registrations": None,
                     "Date": (start + timedelta(days=day)).isoformat(),
                     "Session": slot, "Period": period + 1})
    return rows


## 3. Generate one deterministic demonstration instance
There are 120 synthetic students, 12 exams and 8 available periods across four days. The students are split into three cohorts of 40; each cohort takes exams from its own four-exam block. Students take three or four exams within their cohort.

In [ ]:
SEED, STUDENTS, EXAMS, PERIODS = 42, 120, 12, 8
PARTICLES, ITERATIONS = 32, 100
registrations, course_sizes, conflict_weights = generate_instance(SEED, STUDENTS, EXAMS)
parameter_table = pd.DataFrame([
    {'Parameter':'Random seed','Value':SEED,'Meaning':'Reproducible synthetic instance'},
    {'Parameter':'Students','Value':STUDENTS,'Meaning':'Synthetic enrolment records'},
    {'Parameter':'Exams','Value':EXAMS,'Meaning':'Each exam scheduled once'},
    {'Parameter':'Available periods','Value':PERIODS,'Meaning':'Two periods per day across four days'},
    {'Parameter':'Swarm size','Value':PARTICLES,'Meaning':'PSO particles'},
    {'Parameter':'Iterations','Value':ITERATIONS,'Meaning':'PSO update cycles'},
    {'Parameter':'Total registrations','Value':sum(map(len, registrations)),'Meaning':'Synthetic student-exam enrolments'},
])
display(parameter_table)


## 4. Compare baselines and PSO
All methods receive the same registration instance. Random assignment is an intentionally un-repaired baseline. Greedy and PSO schedules pass through a conflict-aware decoder. Wall time varies by runtime; the fixed-seed objective and audit are the primary comparison.

In [ ]:
timings = {}
t0 = time.perf_counter(); random_result = random_baseline(conflict_weights, PERIODS, SEED + 99); timings['Random assignment'] = time.perf_counter()-t0
t0 = time.perf_counter(); greedy_result = greedy_baseline(conflict_weights, PERIODS); timings['Degree-greedy repair'] = time.perf_counter()-t0
t0 = time.perf_counter(); pso_result = optimize_pso(conflict_weights, PERIODS, SEED + 99, PARTICLES, ITERATIONS); timings['Discrete PSO'] = time.perf_counter()-t0
results = {'Random assignment':random_result,'Degree-greedy repair':greedy_result,'Discrete PSO':pso_result}
rows=[]
for name,result in results.items():
    rows.append({'Method':name,'Student clash pairs':result['clashes'],'Periods used':result['periods_used'],
                 'Load spread':round(result['load_spread'],3),'Fitness':round(result['fitness'],3),
                 'Runtime seconds':round(timings[name],4),'Feasible':result['clashes']==0})
benchmark_table=pd.DataFrame(rows)
display(benchmark_table)


## 5. Final timetable and period loads
Each row below assigns one exam exactly once. Day and session are display labels for the eight available periods.

In [ ]:
best = pso_result
start_date = date(2026, 11, 9)
schedule_rows=[]
for exam, period in enumerate(best['plan']):
    schedule_rows.append({'Exam':f'EX-{exam+1:02d}','Registrations':course_sizes[exam],
        'Day':(start_date+timedelta(days=period//2)).strftime('%A, %d %b %Y'),
        'Session':'09:00-12:00' if period%2==0 else '13:00-16:00','Period':period+1})
schedule_table=pd.DataFrame(schedule_rows).sort_values(['Period','Exam']).reset_index(drop=True)
display(schedule_table)
period_table=(schedule_table.groupby(['Period','Day','Session'],as_index=False)
              .agg(Exams=('Exam','count'),Registrations=('Registrations','sum'))
              .sort_values('Period'))
display(period_table)


## 6. Data and conflict detail
The sample below intentionally shows anonymous synthetic student labels only. Conflict weights count how many synthetic students take each pair of exams; the matrix is symmetric with a zero diagonal.

In [ ]:
registration_table=pd.DataFrame([
    {'Synthetic student':f'S-{i+1:03d}','Exams':', '.join(f'EX-{e+1:02d}' for e in row),'Exam count':len(row)}
    for i,row in enumerate(registrations[:15])])
display(registration_table)
exam_names=[f'EX-{i+1:02d}' for i in range(EXAMS)]
conflict_table=pd.DataFrame(conflict_weights,index=exam_names,columns=exam_names)
display(conflict_table)


## 7. Independent guardrail audit
The audit is run against individual student registrations, independently of the optimizer's weighted objective.

In [ ]:
audit_conflicts=sum(1 for row in registrations for i,left in enumerate(row)
                     for right in row[i+1:] if best['plan'][left]==best['plan'][right])
assert len(best['plan']) == EXAMS, 'Every exam must be assigned exactly once.'
assert all(isinstance(p,int) and 0 <= p < PERIODS for p in best['plan']), 'Period out of bounds.'
assert audit_conflicts == best['clashes'], 'Independent audit must match weighted objective.'
assert len(schedule_table) == EXAMS and schedule_table['Exam'].nunique() == EXAMS
audit_table=pd.DataFrame([
    {'Check':'All exams scheduled once','Result':len(schedule_table)==EXAMS,'Observed':len(schedule_table)},
    {'Check':'All assigned periods in range','Result':all(0<=p<PERIODS for p in best['plan']),'Observed':f'0 to {PERIODS-1}'},
    {'Check':'Independent student clash audit','Result':audit_conflicts==0,'Observed':audit_conflicts},
    {'Check':'Optimizer and audit agree','Result':audit_conflicts==best['clashes'],'Observed':f'{audit_conflicts} = {best["clashes"]}'},
])
display(audit_table)


## 8. Monochrome conflict visualization
A cell is darker when more students take that exam pair. Values are synthetic and are not an institutional timetable.

In [ ]:
fig,ax=plt.subplots(figsize=(8,6))
image=ax.imshow(conflict_weights,cmap='Greys',interpolation='nearest')
ax.set_xticks(range(EXAMS),exam_names,rotation=45,ha='right'); ax.set_yticks(range(EXAMS),exam_names)
ax.set_title('Synthetic student conflict weights'); ax.set_xlabel('Exam'); ax.set_ylabel('Exam')
fig.colorbar(image,ax=ax,label='Shared synthetic students'); plt.tight_layout(); plt.show()


## 9. Export tables
The CSV files are created in the Colab runtime folder. Download them from the Files panel if needed.

In [ ]:
benchmark_table.to_csv('benchmark_results.csv',index=False)
schedule_table.to_csv('optimized_timetable.csv',index=False)
registration_table.to_csv('synthetic_registration_sample.csv',index=False)
conflict_table.to_csv('conflict_weight_matrix.csv')
audit_table.to_csv('guardrail_audit.csv',index=False)
print('Exported: benchmark_results.csv, optimized_timetable.csv, synthetic_registration_sample.csv, conflict_weight_matrix.csv, guardrail_audit.csv')


## 10. Interpretation and limits
A zero-clash schedule means no two exams share a period for any student in this generated sample. It does not prove global optimality: PSO is stochastic and this implementation explores a constrained heuristic search space. Period capacity, rooms, invigilators, accessibility, course precedence and real-world student records are out of scope. A production timetable must add those constraints and have its data owner validate the result before use. Runtime measurements depend on the Colab hardware and session.